In [0]:
# ==============================================================================
# NOTEBOOK: 02_bronze_facts_ingestion
# OBJECTIVE: Ingest Fact CSV batches from UC Volume into Bronze Delta Tables
# ==============================================================================

import os
from datetime import datetime
from pyspark.sql import functions as F

# 1. PATH CONFIGURATIONS & SCHEMAS
volume_base_path = "/Volumes/7trents_data_platform/bronze/raw_data"
target_schema = "7trents_data_platform.bronze"

# Ensure target schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema}")

# Mapping: Source prefix -> Target Bronze Delta table name
fact_sources = {
    "FactWork": "fact_work",
    "FactShift": "fact_shift",
    "FactShrinkage": "fact_shrinkage"
}

# The ordered sequence of daily arrivals
dates_to_process = [
    "01_04_2026",
    "02_04_2026",
    "03_04_2026",
    "04_04_2026",
    "05_04_2026"
]

print("Target Bronze Schema:", target_schema)
print("Processing Dates:", dates_to_process)

In [0]:

# 2. BRONZE INGESTION FUNCTION (READ CSV + AUDIT LINEAGE)
def read_fact_with_audit(source_name, current_date_str):
    file_name = f"{source_name}_{current_date_str}.csv"
    file_path = os.path.join(volume_base_path, current_date_str, file_name)

    print(f"Reading: {file_path}")

    # Read CSV
    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(file_path)
    )

    # Determine load type: Day 1 is FULL, subsequent days are DELTA
    if current_date_str == "01_04_2026":
        load_type_val = "FULL"
    else:
        load_type_val = "DELTA"
    #load_type_val = "FULL" if current_date_str == "01_04_2026" else "DELTA"

    # Append standard metadata audit columns
    df_audit = (
        df
        .withColumn("_source_file", F.lit(file_name))
        .withColumn("_batch_date", F.to_date(F.lit(current_date_str), "dd_MM_yyyy"))
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_load_type", F.lit(load_type_val))
    )

    return df_audit

In [0]:
# 3. WRITE / APPEND FUNCTION WITH IDEMPOTENCY CHECK
def ingest_fact_day(source_name, current_date_str):
    table_name = fact_sources[source_name]
    target_table = f"{target_schema}.{table_name}"
    parsed_date = datetime.strptime(current_date_str, "%d_%m_%Y").date()

    # Idempotency check: Skip if this date batch has already been appended
    if spark.catalog.tableExists(target_table):
        already_loaded = (
            spark.table(target_table)
            .filter(F.col("_batch_date") == F.lit(parsed_date))
            .limit(1)
            .count() > 0
        )
        if already_loaded:
            print(f"-> Batch {current_date_str} for '{table_name}' already exists. Skipping.")
            return

    # Read data from the UC volume
    df = read_fact_with_audit(source_name, current_date_str)

    # Append into Bronze Delta Table (creates table automatically on Day 1)
    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(target_table)
    )
    print(f"✓ Saved {df.count()} rows to '{target_table}' for batch {current_date_str}")

In [0]:
# we have already created now we are just mapping pasted here for testing only

# Mapping: Source prefix -> Target Bronze Delta table name
fact_sources = {
    "FactWork": "fact_work",
    "FactShift": "fact_shift",
    "FactShrinkage": "fact_shrinkage"
}

# The ordered sequence of daily arrivals
dates_to_process = [
    "01_04_2026",
    "02_04_2026",
    "03_04_2026",
    "04_04_2026",
    "05_04_2026"
]

# 4. EXECUTE INGESTION ACROSS ALL DATES AND FACT TABLES
for date_str in dates_to_process:
    print("\n" + "="*50)
    print(f"STARTING BATCH: {date_str}")
    print("="*50)
    for source_prefix in fact_sources.keys():
        ingest_fact_day(source_prefix, date_str)


In [0]:

# 5. POST-LOAD AUDIT & VERIFICATION
print("\n" + "="*50)
print("FINAL BRONZE FACT TABLES VERIFICATION:")
print("="*50)

for table_name in fact_sources.values():
    full_table_name = f"{target_schema}.{table_name}"
    print(f"\nAudit for: {full_table_name}")
    
    df_check = spark.table(full_table_name)
    df_check.groupBy("_batch_date", "_load_type").count().orderBy("_batch_date").show()